In [16]:
# Threshold scan display, channel channel -> MuTrig channel mapping

import json
import numpy as np
import sys
import plotly.graph_objects as go
from plotly.subplots import make_subplots
sys.path.append("../../")
sys.path.append("../../../../midas/python")

import mutrig.eth_selection as eth_ana
import mutrig.ethreshold_scan as eth_scan



filename = "../data/ethreshold_scans/eth_scan_13-10-2026-00.54.json"
filename_output = filename + ".calib.json"

mutrig_channels = range(31)

with open(filename, "r") as f:
    data = json.load(f)

data = eth_ana.read_data(filename)
eths_arr, bad_channels, channels_details = eth_ana.select_rate(data,constrate_value=1e0,ratecut=10, skip=4)
eth_scan.write_config_odb(eths_arr, data['settings'], path = filename_output)


plot_groups = ["Raw","Calibrated"]
mutrig_channels = range(32)

ncols=4 ; nrows=8
fig = make_subplots(
    rows=nrows,
    cols=ncols,
    subplot_titles=plot_groups,
    shared_xaxes=True,
    shared_yaxes=True,
)

all_rates = []
for ch in mutrig_channels:
    therow=int(ch/4)+1,
    thecol=ch%4 + 1,
    if ch >= len(data["th"]) or ch >= len(data["rates"]):
        continue
    if len(data["th"][ch]) == 0 or len(data["rates"][ch]) == 0:
        continue

    th = np.array(data["th"][ch])
    rates = np.array(data["rates"][ch])
    th_calib = data["th"][ch]
    rates_calib = data["rates"][ch]

    selected_threshold = eths_arr[ch]
    selected_rate = 1
    #print(f"selected: {selected_threshold} -> index")

    all_rates.append(min(rates))
    all_rates.append(max(rates))
    
    fig.add_trace(
        go.Scatter(
            x=th,
            y=rates,
            mode="lines+markers",
            name=f"Channel {ch:02d}",
            hovertemplate=(
                f"MuTrig {ch:02d}<br>"
                "Threshold DAC: %{x}<br>"
                "Rate: %{y:.3g}<extra></extra>"
            ),
        ),
        row=int(ch/4)+1,
        col=ch%4 + 1,
    )

    # Add threshold finding result marker
    if (channels_details[ch][0] == "good"):
        print(f"Adding {selected_threshold}, {selected_rate}")
        fig.add_trace(
            go.Scatter(
                x=[selected_threshold],
                y=[selected_rate],
                mode="markers",
                name=f"Selected threshold Ch {ch:02d}",
                #legendgroup=legendgroup,
                showlegend=(row == 1),
                marker=dict(symbol="star", size=16, color="red", line=dict(color="black", width=1)),
                hovertemplate=(
                    f"Selected threshold - Channel {ch:02d}<br>"
                    "Threshold: %{x}<br>Rate: %{y:.3g}<extra></extra>"
                ),
            ),
            row=int(ch/4)+1,
            col=ch%4 + 1,
        )


yrange = [
    np.log10(min(all_rates)+1) - 0.25,
    np.log10(max(all_rates)) + 0.25,
]

fig.update_xaxes(
    title_text="Threshold DAC",
    range=[0, 255],
    tickmode="linear",
    dtick=10,
)

fig.update_yaxes(
    title_text="Rates",
    type="log",
    range=yrange,
)

fig.update_layout(
    title=(
        "Threshold scan"
    ),
    height=900,
    width=1400,
    template="plotly_white",
    hovermode="closest",
    legend=dict(
        groupclick="toggleitem",
        itemclick="toggle",
        itemdoubleclick="toggleothers",
    ),
)

fig.show()

Selecting TTH based on constant target rate of  1.0  Hz
Number of channels:  512
Channel 0 -> Sel=124 | ['good']
Channel 1 -> Sel=107 | ['good']
Channel 2 -> Sel=108 | ['good']
Channel 3 -> Sel=110 | ['good']
Channel 4 -> Sel=124 | ['good']
Channel 5 -> Sel=115 | ['good']
Channel 6 -> Sel=123 | ['good']
Channel 7 -> Sel=107 | ['good']
Channel 8 -> Sel=126 | ['good']
Channel 9 -> Sel=116 | ['good']
Channel 10 -> Sel=89 | ['good']
Channel 11 -> Sel=117 | ['good']
Channel 12 -> Sel=115 | ['good']
Channel 13 -> Sel=100 | ['good']
Channel 14 -> Sel=115 | ['good']
Channel 15 -> Sel=127 | ['good']
Channel 16 -> Sel=113 | ['good']
Channel 17 -> Sel=113 | ['good']
Channel 18 -> Sel=108 | ['good']
Channel 19 -> Sel=120 | ['good']
Channel 20 -> Sel=110 | ['good']
Channel 21 -> Sel=123 | ['good']
Channel 22 -> Sel=106 | ['good']
Channel 23 -> Sel=122 | ['good']
Channel 24 -> Sel=108 | ['good']
Channel 25 -> Sel=108 | ['good']
Channel 26 -> Sel=103 | ['bad', 'ratecut']
Channel 27 -> Sel=129 | ['goo